# Pandas III — Análisis avanzado con dataset real

## Objetivos
- Preparar un dataset real para análisis.
- Hacer checks contables y temporales.
- Dominar `groupby`, `agg`, `transform`, series temporales y `pivot_table`.
- Llegar a reporting útil sin intentar cubrir demasiado.

## Nota de sesión
Este notebook está recortado para que dé tiempo a cerrar lo pendiente de Pandas II y llegar a lo esencial de Pandas III.

---
## 0. Setup

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', '{:,.2f}'.format)


---
## 1. Carga e inspección inicial

In [5]:
path = 'supermarket_sales - Sheet1.csv'
df = pd.read_csv(path)

print('shape:', df.shape)
display(df.head())


print('Columns: ')
for col in df.columns:
    print(col)


shape: (1000, 17)


,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.14,548.97,1/5/2019,13:08,Ewallet,522.83,4.76,26.14,9.10
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.82,80.22,3/8/2019,10:29,Cash,76.40,4.76,3.82,9.60
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.22,340.53,3/3/2019,13:23,Credit card,324.31,4.76,16.22,7.40
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.29,489.05,1/27/2019,20:33,Ewallet,465.76,4.76,23.29,8.40
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.21,634.38,2/8/2019,10:37,Ewallet,604.17,4.76,30.21,5.30


Columns: 
Invoice ID
Branch
City
Customer type
Gender
Product line
Unit price
Quantity
Tax 5%
Total
Date
Time
Payment
cogs
gross margin percentage
gross income
Rating


### Demo guiada 1: inspección rápida

Esta parte la puedes enseñar tú sin convertirla en ejercicio largo.

In [6]:
df.info()

display(df[['Invoice ID', 'City', 'Product line', 'Payment']].nunique())
print('Exact duplicated rows:', df.duplicated().sum())


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Invoice ID               1000 non-null   str    
 1   Branch                   1000 non-null   str    
 2   City                     1000 non-null   str    
 3   Customer type            1000 non-null   str    
 4   Gender                   1000 non-null   str    
 5   Product line             1000 non-null   str    
 6   Unit price               1000 non-null   float64
 7   Quantity                 1000 non-null   int64  
 8   Tax 5%                   1000 non-null   float64
 9   Total                    1000 non-null   float64
 10  Date                     1000 non-null   str    
 11  Time                     1000 non-null   str    
 12  Payment                  1000 non-null   str    
 13  cogs                     1000 non-null   float64
 14  gross margin percentage  1000 non-nu

Invoice ID      1000
City               3
Product line       6
Payment            3
dtype: int64

Exact duplicated rows: 0


In [9]:
df[["Date", "Time"]].head()

,Date,Time
0,1/5/2019,13:08
1,3/8/2019,10:29
2,3/3/2019,13:23
3,1/27/2019,20:33
4,2/8/2019,10:37


---
## 2. Tipos, datetime y checks básicos

Aquí dejamos listo el dataset y hacemos las validaciones mínimas, pero en modo guiado para no comernos la sesión.

### Ejemplo: parseo de fechas y features temporales

In [7]:
df2 = df.copy()
df2['Date'] = pd.to_datetime(df2['Date'])
df2['datetime'] = pd.to_datetime(df2['Date'].dt.strftime('%Y-%m-%d') + ' ' + df2['Time'])
df2['month'] = df2['datetime'].dt.month
df2['weekday'] = df2['datetime'].dt.day_name()
df2['hour'] = df2['datetime'].dt.hour

display(df2[['Date', 'Time', 'datetime', 'month', 'weekday', 'hour']].head())


,Date,Time,datetime,month,weekday,hour
0,2019-01-05,13:08,2019-01-05 13:08:00,1,Saturday,13
1,2019-03-08,10:29,2019-03-08 10:29:00,3,Friday,10
2,2019-03-03,13:23,2019-03-03 13:23:00,3,Sunday,13
3,2019-01-27,20:33,2019-01-27 20:33:00,1,Sunday,20
4,2019-02-08,10:37,2019-02-08 10:37:00,2,Friday,10


In [10]:
for col in df2.columns:
    print(df2[col].dtype, col)

str Invoice ID
str Branch
str City
str Customer type
str Gender
str Product line
float64 Unit price
int64 Quantity
float64 Tax 5%
float64 Total
datetime64[us] Date
str Time
str Payment
float64 cogs
float64 gross margin percentage
float64 gross income
float64 Rating
datetime64[us] datetime
int32 month
str weekday
int32 hour


### Demo guiada 2: categóricas y memoria

In [11]:
mem_before = df2.memory_usage(deep=True).sum()
cat_cols = ['Branch', 'City', 'Customer type', 'Gender', 'Product line', 'Payment']
df2[cat_cols] = df2[cat_cols].astype('category')
mem_after = df2.memory_usage(deep=True).sum()

print('Memory before:', mem_before)
print('Memory after :', mem_after)
print('Reduction %   :', round((1 - mem_after / mem_before) * 100, 2))
display(df2.dtypes)


Memory before: 597725
Memory after : 265435
Reduction %   : 55.59


Invoice ID                            str
Branch                           category
City                             category
Customer type                    category
Gender                           category
Product line                     category
Unit price                        float64
Quantity                            int64
Tax 5%                            float64
Total                             float64
Date                       datetime64[us]
Time                                  str
Payment                          category
cogs                              float64
gross margin percentage           float64
gross income                      float64
Rating                            float64
datetime                   datetime64[us]
month                               int32
weekday                               str
hour                                int32
dtype: object

### Demo guiada 3: sanity checks

In [12]:
tol = 1e-6
fail_total = df2.loc[~np.isclose(df2['Total'], df2['cogs'] + df2['Tax 5%'], atol=tol)]
fail_income = df2.loc[~np.isclose(df2['gross income'], df2['cogs'] * 0.05, atol=tol)]

print('fail_total rows :', fail_total.shape[0])
print('fail_income rows:', fail_income.shape[0])
display(fail_total.head())
display(fail_income.head())


fail_total rows : 0
fail_income rows: 0


,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating,datetime,month,weekday,hour


,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating,datetime,month,weekday,hour


In [15]:
failed_quantity = df2[df2['Quantity'] < 1]
display(failed_quantity.head())

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating,datetime,month,weekday,hour


---
## 3. `groupby` + `agg` para KPIs

Este es uno de los bloques clave de la sesión.

### Ejemplo: KPIs por ciudad

In [16]:
city_kpis = df2.groupby('City', as_index=False).agg(
    total_sales=('Total', 'sum'),
    tx_count=('Invoice ID', 'count'),
    avg_ticket=('Total', 'mean'),
    avg_rating=('Rating', 'mean'),
    total_profit=('gross income', 'sum')
).sort_values('total_sales', ascending=False)

display(city_kpis)


,City,total_sales,tx_count,avg_ticket,avg_rating,total_profit
1,Naypyitaw,"110,568.71",328,337.10,7.07,"5,265.18"
2,Yangon,"106,200.37",340,312.35,7.03,"5,057.16"
0,Mandalay,"106,197.67",332,319.87,6.82,"5,057.03"


In [17]:
df2["Invoice ID"].is_unique

True

### Ejercicio 1
Crea tablas KPI con las mismas métricas por:
1. `Branch`
2. `Product line`
3. `Payment`

In [20]:
branch_kpis = df2.groupby('Branch', as_index=False).agg(
    total_sales=('Total', 'sum'),
    tx_count=('Invoice ID', 'count'),
    avg_ticket=('Total', 'mean'),
    avg_rating=('Rating', 'mean'),
    total_profit=('gross income', 'sum')
).sort_values('total_sales', ascending=False)

display(branch_kpis)

product_line_kpis = df2.groupby('Product line', as_index=False).agg(
    total_sales=('Total', 'sum'),
    tx_count=('Invoice ID', 'count'),
    avg_ticket=('Total', 'mean'),
    avg_rating=('Rating', 'mean'),
    total_profit=('gross income', 'sum')
).sort_values('total_sales', ascending=False)

display(product_line_kpis)

payment_kpis = df2.groupby('Payment', as_index=False).agg(
    total_sales=('Total', 'sum'),
    tx_count=('Invoice ID', 'count'),
    avg_ticket=('Total', 'mean'),
    avg_rating=('Rating', 'mean'),
    total_profit=('gross income', 'sum')
).sort_values('total_sales', ascending=False)

display(payment_kpis)

,Branch,total_sales,tx_count,avg_ticket,avg_rating,total_profit
2,C,"110,568.71",328,337.10,7.07,"5,265.18"
0,A,"106,200.37",340,312.35,7.03,"5,057.16"
1,B,"106,197.67",332,319.87,6.82,"5,057.03"


,Product line,total_sales,tx_count,avg_ticket,avg_rating,total_profit
2,Food and beverages,"56,144.84",174,322.67,7.11,"2,673.56"
5,Sports and travel,"55,122.83",166,332.07,6.92,"2,624.90"
0,Electronic accessories,"54,337.53",170,319.63,6.92,"2,587.50"
1,Fashion accessories,"54,305.89",178,305.09,7.03,"2,585.99"
4,Home and lifestyle,"53,861.91",160,336.64,6.84,"2,564.85"
3,Health and beauty,"49,193.74",152,323.64,7.00,"2,342.56"


,Payment,total_sales,tx_count,avg_ticket,avg_rating,total_profit
0,Cash,"112,206.57",344,326.18,6.97,"5,343.17"
2,Ewallet,"109,993.11",345,318.82,6.95,"5,237.77"
1,Credit card,"100,767.07",311,324.01,7.00,"4,798.43"


---
## 4. `transform`

`transform` te permite devolver a cada fila una métrica calculada a nivel grupo.

### Ejemplo: share del ticket dentro de su ciudad

In [21]:
fs = df2.copy()
fs['city_total_sales'] = fs.groupby('City')['Total'].transform('sum')
fs['ticket_share_city'] = fs['Total'] / fs['city_total_sales']

display(fs[['City', 'Total', 'city_total_sales', 'ticket_share_city']].head())


,City,Total,city_total_sales,ticket_share_city
0,Yangon,548.97,"106,200.37",0.01
1,Naypyitaw,80.22,"110,568.71",0.00
2,Yangon,340.53,"106,200.37",0.00
3,Yangon,489.05,"106,200.37",0.00
4,Yangon,634.38,"106,200.37",0.01


### Ejercicio 2
1. Crea `product_line_total_sales` con `transform`.
2. Crea `ticket_share_product_line`.
3. Ordena por `Product line` y `ticket_share_product_line` descendente.

In [22]:
fs2 = df2.copy()
fs2['product_line_total_sales'] = fs2.groupby('Product line')['Total'].transform('sum')
fs2['ticket_share_product_line'] = fs2['Total'] / fs2['product_line_total_sales']

print("Ordered by product line descending:")

display(fs2.sort_values('Product line', ascending=False)[['Product line', 'Total', 'product_line_total_sales', 'ticket_share_product_line']].head())

Ordered by product line descending:


,Product line,Total,product_line_total_sales,ticket_share_product_line
500,Sports and travel,77.67,"55,122.83",0.00
727,Sports and travel,732.27,"55,122.83",0.01
721,Sports and travel,856.45,"55,122.83",0.02
213,Sports and travel,146.22,"55,122.83",0.00
214,Sports and travel,217.63,"55,122.83",0.00


---
## 5. Series temporales

Nos quedamos con la parte más útil: agregación temporal y patrones.

### Ejemplo: serie diaria y rolling de 7 días

In [ ]:
ts = df2.set_index('datetime').sort_index()
daily_sales = ts['Total'].resample('D').sum().fillna(0)
daily_roll7 = daily_sales.rolling(7, min_periods=1).mean()
#El rolling calcula una serie de datos de una ventana movil de 7 dias en este caso, min periods para asegurarse que al menos tiene un periodo de datos
#En resumen, daily_roll7 contiene los promedios de los últimos 7 días de ventas en la serie de datos daily_sales.
display(pd.DataFrame({'daily_sales': daily_sales, 'roll7': daily_roll7}).head(10))


,daily_sales,roll7
datetime,,
2019-01-01,"4,745.18","4,745.18"
2019-01-02,"1,945.50","3,345.34"
2019-01-03,"2,078.13","2,922.94"
2019-01-04,"1,623.69","2,598.13"
2019-01-05,"3,536.68","2,785.84"
2019-01-06,"3,614.20","2,923.90"
2019-01-07,"2,834.24","2,911.09"
2019-01-08,"5,293.73","2,989.45"
2019-01-09,"3,021.34","3,143.15"


In [28]:
display(ts.head(2))  
print(daily_sales.head(2))

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating,month,weekday,hour
datetime,,,,,,,,,,,,,,,,,,,,
2019-01-01 10:39:00,765-26-6951,A,Yangon,Normal,Male,Sports and travel,72.61,6,21.78,457.44,2019-01-01,10:39,Credit card,435.66,4.76,21.78,6.90,1,Tuesday,10
2019-01-01 11:36:00,746-04-1077,B,Mandalay,Member,Female,Food and beverages,84.63,10,42.31,888.62,2019-01-01,11:36,Credit card,846.30,4.76,42.31,9.00,1,Tuesday,11


datetime
2019-01-01   4,745.18
2019-01-02   1,945.50
Freq: D, Name: Total, dtype: float64


### Ejercicio 3
1. Construye una serie semanal (`W`) de ventas.
2. Calcula la media móvil de 4 semanas.
3. Devuelve un DataFrame con `weekly_sales` y `rolling_4w`.

In [ ]:
#resample('W') cambia la frecuencia de los datos de diarios a semanales y sum() calcula la suma de los valores en cada intervalo semanal.
weekly_sales = ts['Total'].resample('W').sum().fillna(0)
rolling_4w = weekly_sales.rolling(4, min_periods=2).mean().fillna(0)
#fillna(0) rellena los datos faltantes con 0 para que no haya errores en los calculos posteriores (en este caso si no se pone, en uno de los casos da NaN al no tener dato)
display(pd.DataFrame({'weekly_sales': weekly_sales, 'roll4w': rolling_4w}).head(10))



,weekly_sales,roll4w
datetime,,
2019-01-06,"17,543.39",0.00
2019-01-13,"24,461.20","21,002.29"
2019-01-20,"28,693.36","23,565.98"
2019-01-27,"29,286.88","24,996.21"
2019-02-03,"28,360.45","27,700.47"
2019-02-10,"27,101.83","28,360.63"
2019-02-17,"25,563.59","27,578.19"
2019-02-24,"17,328.66","24,588.63"
2019-03-03,"29,219.72","24,803.45"


### Ejercicio 4
1. Calcula ventas totales por `weekday`.
2. Calcula ventas totales por `hour`.
3. Encuentra la combinación `weekday` + `hour` con mayor venta total.

In [32]:
weekday_sales = df2.groupby('weekday', as_index=False).agg(total_sales=('Total', 'sum')).sort_values('total_sales', ascending=False)
hour_sales = df2.groupby('hour', as_index=False).agg(total_sales=('Total', 'sum')).sort_values('total_sales', ascending=False)
weekday_hour = (
df2.groupby(['weekday', 'hour'], as_index=False)
       .agg(total_sales=('Total', 'sum'))
       .sort_values('total_sales', ascending=False)
)

display(weekday_sales)
display(hour_sales.head())
display(weekday_hour.head(1))



,weekday,total_sales
2,Saturday,"56,120.81"
5,Tuesday,"51,482.25"
4,Thursday,"45,349.25"
3,Sunday,"44,457.89"
0,Friday,"43,926.34"
6,Wednesday,"43,731.14"
1,Monday,"37,899.08"


,hour,total_sales
9,19,"39,699.51"
3,13,"34,723.23"
0,10,"31,421.48"
5,15,"31,179.51"
4,14,"30,828.40"


,weekday,hour,total_sales
64,Tuesday,19,"9,198.67"


---
## 6. Pivot tables

Este bloque suele dar muchísimo retorno práctico en clase.

### Ejemplo: City × Payment

In [ ]:
pv = pd.pivot_table(df2, values='Total', index='City', columns='Payment', aggfunc='sum').fillna(0)
pv['Total'] = pv.sum(axis=1)
pv = pv.sort_values('Total', ascending=False)

display(pv)
#pivot table son tablas que vienen agregadas
#o tabla de resumen.
#esto crea una tabla a partir del df2 que agrupa los datos por ciudad y el metodo sum() calcula la suma de los valores en cada columna.



Payment,Cash,Credit card,Ewallet,Total
City,,,,
Naypyitaw,"43,085.86","30,327.47","37,155.38","110,568.71"
Yangon,"33,781.25","33,094.75","39,324.37","106,200.37"
Mandalay,"35,339.46","37,344.86","33,513.35","106,197.67"


### Ejercicio 5
Crea estos pivots:
1. `Branch` × `Product line` con suma de `Total`.
2. `City` × `Gender` con media de `Rating`.
3. `Branch` × `Payment` con suma de `gross income`.

In [36]:
pv1 = pd.pivot_table(df2, values='Total', index='Branch', columns='Product line', aggfunc='sum').fillna(0)
pv2 = pd.pivot_table(df2, values='Rating', index='City', columns='Gender', aggfunc='mean').fillna(0)
pv3 = pd.pivot_table(df2, values='gross income', index='Branch', columns='Payment', aggfunc='sum').fillna(0)
display(pv1)
display(pv2)
display(pv3)



Product line,Electronic accessories,Fashion accessories,Food and beverages,Health and beauty,Home and lifestyle,Sports and travel
Branch,,,,,,
A,"18,317.11","16,332.51","17,163.10","12,597.75","22,417.20","19,372.70"
B,"17,051.44","16,413.32","15,214.89","19,980.66","17,549.16","19,988.20"
C,"18,968.97","21,560.07","23,766.85","16,615.33","13,895.55","15,761.93"


Gender,Female,Male
City,,
Mandalay,6.88,6.76
Naypyitaw,7.16,6.97
Yangon,6.84,7.20


Payment,Cash,Credit card,Ewallet
Branch,,,
A,"1,608.63","1,575.94","1,872.59"
B,"1,682.83","1,778.33","1,595.87"
C,"2,051.71","1,444.16","1,769.30"


---
## 7. Extensión opcional

Si sobra tiempo, puedes cerrar la sesión con un mini reporte.

### Opcional: `build_report`
Implementa `build_report(df)` para que devuelva un diccionario con estas tablas:
- `sales_by_city_branch`
- `profit_by_product_line`
- `ticket_band_mix`
- `weekday_hour_heatmap`

In [38]:
def build_report(data: pd.DataFrame) -> dict:
    sales_by_city_branch = data.groupby(['City', 'Branch'], as_index=False).agg(total_sales=('Total', 'sum')).sort_values('total_sales', ascending=False)
    profit_by_product_line = data.groupby('Product line', as_index=False).agg(total_profit=('gross income', 'sum')).sort_values('total_profit', ascending=False)

    data["ticket_band"] = pd.cut(
        data['Total'],
        bins=[0, 100, 300, 1000, np.inf],
        labels=['Low', 'Medium', 'High', 'Very High']
    )

    ticket_band_mix = data.groupby('ticket_band', as_index=False).agg(
        total_sales=('Total', 'sum'),
        tx_count=('Invoice ID', 'count'),
        avg_rating=('Rating', 'mean')
    ).sort_values('total_sales', ascending=False)

    weekday_hour_heatmap = pd.pivot_table(
        data,
        values='Total',
        index='weekday',
        columns='hour',
        aggfunc='mean',
        fill_value=0
    )
    return {
    'sales_by_city_branch': sales_by_city_branch,
    'profit_by_product_line': profit_by_product_line,
    'ticket_band_mix': ticket_band_mix,
    'weekday_hour_heatmap': weekday_hour_heatmap
    }





report = build_report(df2)
print(report.keys())
for key, df_report in report.items():
    print(f'\nReport: {key}')

display(df_report.head(7))


dict_keys(['sales_by_city_branch', 'profit_by_product_line', 'ticket_band_mix', 'weekday_hour_heatmap'])

Report: sales_by_city_branch

Report: profit_by_product_line

Report: ticket_band_mix

Report: weekday_hour_heatmap


hour,10,11,12,13,14,15,16,17,18,19,20
weekday,,,,,,,,,,,
Friday,315.09,240.84,215.92,426.52,369.02,205.34,309.23,315.57,288.66,421.23,343.54
Monday,311.54,319.18,337.58,375.86,334.06,285.63,293.19,289.02,255.10,324.83,238.26
Saturday,206.00,457.24,272.83,406.40,485.62,289.25,272.39,422.48,374.38,396.41,227.51
Sunday,369.76,430.62,429.84,286.64,359.70,243.81,373.54,379.03,242.48,350.80,266.28
Thursday,382.51,293.53,233.10,216.19,391.00,359.42,360.60,270.44,335.73,318.51,452.52
Tuesday,286.59,261.19,339.92,304.81,330.21,501.44,425.72,330.61,201.47,328.52,324.97
Wednesday,317.73,417.08,231.31,336.72,321.26,285.64,322.96,310.13,249.05,297.16,309.20


In [40]:
#El anterior pero con explicaciones

def build_report(data: pd.DataFrame) -> dict:
    # Agrupa los datos del DataFrame por ciudad y sucursal, calcula la suma de las ventas totales para cada combinación y ordena los resultados por la suma de ventas en orden descendente.
    sales_by_city_branch = data.groupby(['City', 'Branch'], as_index=False).agg(total_sales=('Total', 'sum')).sort_values('total_sales', ascending=False)
    
    # Agrupa los datos del DataFrame por línea de producto, calcula la suma de la ganancia total para cada línea y ordena los resultados por la suma de la ganancia en orden descendente.
    profit_by_product_line = data.groupby('Product line', as_index=False).agg(total_profit=('gross income', 'sum')).sort_values('total_profit', ascending=False)

    # Crea una nueva columna 'ticket_band' en el DataFrame, que clasifica las ventas en diferentes rangos utilizando la función pd.cut().
    data["ticket_band"] = pd.cut(
        data['Total'],
        bins=[0, 100, 300, 1000, np.inf],
        labels=['Low', 'Medium', 'High', 'Very High']
    )

    # Agrupa los datos del DataFrame por la columna 'ticket_band', calcula la suma de las ventas totales, el número de facturas y el promedio de la calificación para cada rango y ordena los resultados por la suma de ventas en orden descendente.
    ticket_band_mix = data.groupby('ticket_band', as_index=False).agg(
        total_sales=('Total', 'sum'),
        tx_count=('Invoice ID', 'count'),
        avg_rating=('Rating', 'mean')
    ).sort_values('total_sales', ascending=False)

    # Crea una tabla de resumen utilizando la función pd.pivot_table(), que muestra la media de las ventas por día de la semana y hora.
    weekday_hour_heatmap = pd.pivot_table(
        data,
        values='Total',
        index='weekday',
        columns='hour',
        aggfunc='mean',
        fill_value=0
    )

    # Devuelve un diccionario que contiene los resultados de las agrupaciones y cálculos realizados.
    return {
        'sales_by_city_branch': sales_by_city_branch,
        'profit_by_product_line': profit_by_product_line,
        'ticket_band_mix': ticket_band_mix,
        'weekday_hour_heatmap': weekday_hour_heatmap
    }

# Llama a la función build_report() con el DataFrame df2 como argumento y asigna el resultado a la variable report.
report = build_report(df2)

# Imprime las claves del diccionario report.
print(report.keys())

# Itera sobre cada clave del diccionario report y muestra la primera parte del resultado utilizando la función display().
for key, df_report in report.items():
    print(f'\nReport: {key}')
    display(df_report.head(7))



dict_keys(['sales_by_city_branch', 'profit_by_product_line', 'ticket_band_mix', 'weekday_hour_heatmap'])

Report: sales_by_city_branch


,City,Branch,total_sales
1,Naypyitaw,C,"110,568.71"
2,Yangon,A,"106,200.37"
0,Mandalay,B,"106,197.67"



Report: profit_by_product_line


,Product line,total_profit
2,Food and beverages,"2,673.56"
5,Sports and travel,"2,624.90"
0,Electronic accessories,"2,587.50"
1,Fashion accessories,"2,585.99"
4,Home and lifestyle,"2,564.85"
3,Health and beauty,"2,342.56"



Report: ticket_band_mix


,ticket_band,total_sales,tx_count,avg_rating
2,High,"230,719.60",422,6.93
1,Medium,"69,923.46",361,7.15
0,Low,"13,112.25",208,6.80
3,Very High,"9,211.44",9,6.20



Report: weekday_hour_heatmap


hour,10,11,12,13,14,15,16,17,18,19,20
weekday,,,,,,,,,,,
Friday,315.09,240.84,215.92,426.52,369.02,205.34,309.23,315.57,288.66,421.23,343.54
Monday,311.54,319.18,337.58,375.86,334.06,285.63,293.19,289.02,255.10,324.83,238.26
Saturday,206.00,457.24,272.83,406.40,485.62,289.25,272.39,422.48,374.38,396.41,227.51
Sunday,369.76,430.62,429.84,286.64,359.70,243.81,373.54,379.03,242.48,350.80,266.28
Thursday,382.51,293.53,233.10,216.19,391.00,359.42,360.60,270.44,335.73,318.51,452.52
Tuesday,286.59,261.19,339.92,304.81,330.21,501.44,425.72,330.61,201.47,328.52,324.97
Wednesday,317.73,417.08,231.31,336.72,321.26,285.64,322.96,310.13,249.05,297.16,309.20
